# 15: Training on current generators (E7)

E6 showed that detectors trained on 2021-2023 generators miss many images from 2024-2026 generators. Here we train the same two kinds of detector on **recent data** and test them on exactly the same images as in E6:

- **Training data:** five files of the OpenFake *training* split (about 20,000 images): real photographs from LAION and Pexels, generated images from about 70 generators such as SD 3.5, Flux.1, Midjourney 6, DALL-E 3 and GPT Image 1.
- **Model selection:** one file of the OpenFake *validation* split.
- **Test:** the E6 OpenFake test images (generators *not* in the training split, real photographs from DOCCI and ImageNet), the 100 WhatsApp pairs, and our GenImage test set.

Bias control follows notebook 03: same centre crop; images that are not JPEG files get one JPEG compression at a quality drawn from the real training images. Only the training data changes; the models and settings are those of M1 and M2.

GPU runtime needed; about 30 GB is downloaded. Total time about 60 minutes. Processed arrays are saved to Drive, so a restart skips the slow part.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub

import io
import json
import os
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, list_repo_files
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50
from transformers import CLIPVisionModelWithProjection

ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
DATA = ROOT / "processed_openfake_train"
DOWNLOADS = Path("/content/openfake_train")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
TRAIN_FILES = 5
VAL_FILES = 1
SIZE = 224
SEED = 42
RUN_NAME = "e7_m1_openfake"
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
CLIP_NAME = "openai/clip-vit-large-patch14"

torch.manual_seed(SEED)
np.random.seed(SEED)

## Helper functions (same as notebook 03)

In [ ]:
STD_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
])


def estimate_quality(img):
    luma = np.array(img.quantization[0])
    best_quality, best_error = 0, np.inf
    for quality in range(1, 101):
        scale = 5000 // quality if quality < 50 else 200 - 2 * quality
        table = np.clip((STD_LUMA * scale + 50) // 100, 1, 255)
        error = np.abs(table - luma).sum()
        if error < best_error:
            best_quality, best_error = quality, error
    return best_quality


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))


def jpeg_roundtrip(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=int(quality))
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

## Download and process the training data (about 40 minutes, only the first time)

Files are chosen evenly across the training split. Each file is processed and deleted before the next one is downloaded, to save disk space. Non-JPEG images are first stored uncompressed; the JPEG step is applied afterwards, once the real images' quality distribution is known.

In [ ]:
def process_file(path, split_name):
    pf = pq.ParquetFile(path)
    records, crops = [], []
    for g in range(pf.num_row_groups):
        t = pf.read_row_group(g, columns=["image", "label", "model"])
        for item, label, model in zip(t["image"].to_pylist(), t["label"].to_pylist(), t["model"].to_pylist()):
            img = Image.open(io.BytesIO(item["bytes"]))
            if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < SIZE:
                continue
            records.append({"split": split_name, "source": model, "label": label, "format": img.format,
                            "jpeg_quality": estimate_quality(img) if img.format == "JPEG" else np.nan})
            crops.append(np.asarray(center_crop_aligned(img.convert("RGB"))))
    return pd.DataFrame(records), np.stack(crops)


if not (DATA / "meta.csv").exists():
    all_files = sorted(f for f in list_repo_files("ComplexDataLab/OpenFake", repo_type="dataset") if f.startswith("core/"))
    train_files = [f for f in all_files if "/train-" in f]
    val_files = [f for f in all_files if "/validation-" in f]
    picks = [("train", train_files[i]) for i in np.linspace(0, len(train_files) - 1, TRAIN_FILES).astype(int)]
    picks += [("val", f) for f in val_files[:VAL_FILES]]

    metas, arrays = [], []
    DOWNLOADS.mkdir(exist_ok=True)
    for split_name, name in picks:
        start = time.time()
        path = hf_hub_download("ComplexDataLab/OpenFake", name, repo_type="dataset", local_dir=DOWNLOADS)
        m, a = process_file(path, split_name)
        metas.append(m)
        arrays.append(a)
        os.remove(path)
        print(f"{name}: {len(m)} usable images ({time.time() - start:.0f}s)")
    meta = pd.concat(metas, ignore_index=True)
    images = np.concatenate(arrays)
    del arrays

    # balance real and generated images within each split
    keep = []
    for split_name, group in meta.groupby("split"):
        n = min((group["label"] == "real").sum(), (group["label"] == "fake").sum())
        keep += group[group["label"] == "real"].sample(n=n, random_state=SEED).index.tolist()
        keep += group[group["label"] == "fake"].sample(n=n, random_state=SEED).index.tolist()
    keep = sorted(keep)
    meta, images = meta.loc[keep].reset_index(drop=True), images[keep]

    # bias control: non-JPEG images get one JPEG compression at a quality drawn from the real training images
    real_q = meta.loc[(meta["label"] == "real") & (meta["split"] == "train"), "jpeg_quality"].dropna().to_numpy()
    rng = np.random.default_rng(SEED)
    meta["compressed_by_us"] = meta["format"] != "JPEG"
    for i in meta.index[meta["compressed_by_us"]]:
        q = int(rng.choice(real_q))
        images[i] = np.asarray(jpeg_roundtrip(Image.fromarray(images[i]), q))
        meta.loc[i, "jpeg_quality"] = q

    DATA.mkdir(exist_ok=True)
    np.save(DATA / "images.npy", images)
    meta.to_csv(DATA / "meta.csv", index=False)
    del images

meta = pd.read_csv(DATA / "meta.csv")
images = np.load(DATA / "images.npy", mmap_mode="r")
meta["y"] = (meta["label"] == "fake").astype(int)
print(meta.groupby(["split", "label"]).size().unstack())
print("\nJPEG quality after bias control (should be similar for real and generated):")
display(meta.groupby(["split", "label"])["jpeg_quality"].describe()[["mean", "50%", "min", "max"]].round(1))
print(f"Generators in training data: {meta[meta['y'] == 1]['source'].nunique()}")

## Test data from E6

The E6 crops are reused unchanged. Earlier predictions of the 2023-trained detectors are read from the E6 result files, so old and new detectors are compared on identical images.

In [ ]:
of_images = np.load(ROOT / "processed_openfake" / "images.npy")
of_old = pd.read_csv(RESULTS / "e6_openfake_predictions.csv")
of_y = of_old["y"].to_numpy()

wa_old = pd.read_csv(RESULTS / "e6_whatsapp_predictions.csv")
wa_y = wa_old["y"].to_numpy()
received_dir = Path("/content/received") if (ROOT / "whatsapp" / "received.zip").exists() else ROOT / "whatsapp" / "received"
if (ROOT / "whatsapp" / "received.zip").exists():
    import zipfile
    zipfile.ZipFile(ROOT / "whatsapp" / "received.zip").extractall(received_dir)
received_paths = {p.name: p for p in received_dir.rglob("*")}
wa_before = np.stack([np.asarray(center_crop_aligned(Image.open(ROOT / "whatsapp" / "original" / f).convert("RGB"))) for f in wa_old["file"]])
wa_after = np.stack([np.asarray(center_crop_aligned(Image.open(received_paths[f]).convert("RGB"))) for f in wa_old["received_file"]])

gi_images = np.load(ROOT / "processed" / "test_images.npy", mmap_mode="r")
gi_y = (pd.read_csv(ROOT / "processed" / "test_meta.csv")["label"] == "fake").astype(int).to_numpy()
print(f"OpenFake test: {len(of_y)}, WhatsApp pairs: {len(wa_y)}, GenImage test: {len(gi_y)}")

## New M1: ResNet-50 trained on OpenFake

In [ ]:
class ImageArrayDataset(Dataset):
    def __init__(self, images, rows, labels):
        self.images, self.rows, self.labels = images, rows, labels

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return torch.from_numpy(np.array(self.images[self.rows[i]])), self.labels[i]


def loader(images, rows, labels, shuffle=False):
    dataset = ImageArrayDataset(images, rows, labels)
    return DataLoader(dataset, batch_size=BATCH_SIZE if shuffle else BATCH_SIZE * 2, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True)


train_rows = meta.index[meta["split"] == "train"].to_numpy()
val_rows = meta.index[meta["split"] == "val"].to_numpy()
y_all = meta["y"].to_numpy()
train_loader = loader(images, train_rows, y_all[train_rows], shuffle=True)
val_loader = loader(images, val_rows, y_all[val_rows])

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def to_model_input(batch, augment=False):
    x = batch.to(device, non_blocking=True).permute(0, 3, 1, 2).float() / 255
    if augment:
        flip = torch.rand(x.shape[0], device=device) < 0.5
        x[flip] = x[flip].flip(-1)
    return (x - MEAN) / STD


model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)


@torch.no_grad()
def predict_m1(data):
    model.eval()
    if isinstance(data, DataLoader):
        batches = (b for b, _ in data)
    else:
        batches = (torch.from_numpy(np.array(data[s:s + BATCH_SIZE * 2])) for s in range(0, len(data), BATCH_SIZE * 2))
    probs = []
    for batch in batches:
        with torch.autocast(device.type, enabled=use_amp):
            probs.append(torch.sigmoid(model(to_model_input(batch)).squeeze(1).float()).cpu())
    return torch.cat(probs).numpy()

In [ ]:
CKPT = ROOT / "checkpoints" / f"{RUN_NAME}.pt"
history, best_val_loss = [], float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    start, running = time.time(), 0.0
    for batch, labels in train_loader:
        x = to_model_input(batch, augment=True)
        y = labels.float().to(device)
        with torch.autocast(device.type, enabled=use_amp):
            loss = loss_fn(model(x).squeeze(1), y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * len(y)
    val_probs = predict_m1(val_loader)
    val_loss = float(log_loss(y_all[val_rows], val_probs, labels=[0, 1]))
    history.append({"epoch": epoch, "train_loss": running / len(train_rows), "val_loss": val_loss,
                    "val_auc": float(roc_auc_score(y_all[val_rows], val_probs))})
    print(f"Epoch {epoch}: train loss {history[-1]['train_loss']:.4f} | val loss {val_loss:.4f} | "
          f"val AUC {history[-1]['val_auc']:.4f} | {time.time() - start:.0f}s")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT)
        print("  saved as best so far")
model.load_state_dict(torch.load(CKPT, map_location=device))

## New M2: CLIP features + logistic regression trained on OpenFake

In [ ]:
dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def clip_features(data, rows=None):
    rows = np.arange(len(data)) if rows is None else rows
    out = []
    for s in range(0, len(rows), 128):
        x = torch.from_numpy(np.array(data[rows[s:s + 128]])).to(device).permute(0, 3, 1, 2).float() / 255
        if x.shape[-1] != CLIP_SIZE:
            x = F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False)
        emb = clip(pixel_values=((x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float()
        out.append(F.normalize(emb, dim=1).cpu().numpy())
    return np.concatenate(out)


feat_file = DATA / "clip_features.npy"
if not feat_file.exists():
    np.save(feat_file, clip_features(images))
feats = np.load(feat_file)

search = []
for C in [0.01, 0.1, 1, 10, 100]:
    c = LogisticRegression(C=C, max_iter=5000).fit(feats[train_rows], y_all[train_rows])
    search.append({"C": C, "val_loss": float(log_loss(y_all[val_rows], c.predict_proba(feats[val_rows])[:, 1]))})
    print(f"C={C:<6} val loss {search[-1]['val_loss']:.4f}")
best_C = min(search, key=lambda r: r["val_loss"])["C"]
clf = LogisticRegression(C=best_C, max_iter=5000).fit(feats[train_rows], y_all[train_rows])
print("Best C:", best_C)


def predict_m2(data):
    return clf.predict_proba(clip_features(data))[:, 1]

## Old vs new detectors on the same test images

In [ ]:
def metrics(y, p):
    return {"real_called_real": float((p[y == 0] < 0.5).mean()), "fake_detected": float((p[y == 1] >= 0.5).mean()),
            "auc": float(roc_auc_score(y, p))}


new = {
    "openfake": {"M1": predict_m1(of_images), "M2": predict_m2(of_images)},
    "whatsapp_before": {"M1": predict_m1(wa_before), "M2": predict_m2(wa_before)},
    "whatsapp_after": {"M1": predict_m1(wa_after), "M2": predict_m2(wa_after)},
    "genimage": {"M1": predict_m1(gi_images), "M2": predict_m2(gi_images)},
}
old = {
    "openfake": {"M1": of_old["M1 ResNet-50"].to_numpy(), "M2": of_old["M2 CLIP"].to_numpy()},
    "whatsapp_before": {"M1": wa_old["M1 ResNet-50 before"].to_numpy(), "M2": wa_old["M2 CLIP before"].to_numpy()},
    "whatsapp_after": {"M1": wa_old["M1 ResNet-50 after"].to_numpy(), "M2": wa_old["M2 CLIP after"].to_numpy()},
    "genimage": {"M1": pd.read_csv(RESULTS / "m1_resnet50_test_predictions.csv")["prob_fake"].to_numpy(),
                 "M2": pd.read_csv(RESULTS / "m2_clip_test_predictions.csv")["prob_fake"].to_numpy()},
}
labels = {"openfake": of_y, "whatsapp_before": wa_y, "whatsapp_after": wa_y, "genimage": gi_y}

rows = []
for test in new:
    for m in ["M1", "M2"]:
        for trained, probs in [("GenImage (2021-23)", old), ("OpenFake (2022-25)", new)]:
            rows.append({"test": test, "model": m, "trained_on": trained, **metrics(labels[test], probs[test][m])})
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / "e7_summary.csv", index=False)
json.dump({"best_C": best_C, "clip_search": search, "m1_history": history}, open(RESULTS / f"{RUN_NAME}.json", "w"), indent=2)
display(summary.pivot_table(index=["test", "model"], columns="trained_on", values="auc").round(3))

In [ ]:
per_source = of_old[["source", "y", "release_date", "format"]].copy()
for m in ["M1", "M2"]:
    per_source[f"{m} old"] = (old["openfake"][m] >= 0.5) == of_y
    per_source[f"{m} new"] = (new["openfake"][m] >= 0.5) == of_y
    of_old[f"{m} new"] = new["openfake"][m]
of_old.to_csv(RESULTS / "e7_openfake_predictions.csv", index=False)
by_source = per_source.groupby("source")[["M1 old", "M1 new", "M2 old", "M2 new"]].mean()
by_source.to_csv(RESULTS / "e7_openfake_per_source.csv")
print("Correct rate per OpenFake test source (old = trained on GenImage, new = trained on OpenFake):")
display(by_source.round(3))

## Charts

In [ ]:
tests = [("genimage", "GenImage test\n(2021-23 generators)"), ("openfake", "OpenFake test\n(2024-26 generators)"),
         ("whatsapp_after", "OpenFake after WhatsApp\n(100 images)")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), sharey=True)
x, w = np.arange(2), 0.38
for ax, (test, title) in zip(axes, tests):
    part = summary[summary["test"] == test]
    for k, (trained, color) in enumerate([("GenImage (2021-23)", "#a9b4c2"), ("OpenFake (2022-25)", "#2a78d6")]):
        values = part[part["trained_on"] == trained].set_index("model").loc[["M1", "M2"], "auc"] * 100
        bars = ax.bar(x + (k - 0.5) * w, values, w - 0.03, color=color, label=f"Trained on {trained}")
        ax.bar_label(bars, fmt="%.0f", fontsize=8)
    ax.set_xticks(x, ["M1 ResNet-50", "M2 CLIP"])
    ax.set_title(title, fontsize=10)
    ax.axhline(50, color="#999999", linewidth=1, linestyle="--")
    ax.grid(axis="y", color="#e5e5e5")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("ROC-AUC x 100")
axes[0].set_ylim(0, 105)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.savefig(RESULTS / "e7_then_now.png", dpi=120)
plt.show()